# 멀티턴 대화와 Streamlit 세션 상태

여러 번 주고받은 말을 목록에 저장하고 다음 요청의 입력에 함께 넣습니다. 먼저 파이썬 목록으로 대화의 순서를 확인하고, 같은 목록을 Streamlit 화면에서 유지하는 방법을 익힙니다.

11장의 대화 이력 실습에는 8시간을 씁니다. 처음에는 한 사용자의 정정 범위를 읽고, 다음에는 별도 대화의 질문과 실제 입력을 대조합니다. 12~13장의 화면 실습 16시간은 서버 실행, 작은 에코 화면에서 재실행 관찰, 모델 연결, 두 탭 비교, 처음 보는 자료의 인수 시험까지 이어집니다. 활동지를 새 탭에 열어 실제 응답과 판정 근거를 기록하세요. [공지 챗봇 활동지](practice.md)

전체 실습 묶음을 내려받았다면 압축을 모두 풀고 이 노트북이 들어 있는 폴더에서 시작하세요. 노트북 하나만 받은 경우에는 새 수업 폴더에서 여세요. 호출 함수와 완성 화면을 만드는 셀은 그 폴더에 수업용 파일 두 개를 씁니다. 관찰용 에코 화면은 별도 이름으로 저장합니다. Python 3.10 이상 커널과 앞 단원의 OpenAI API 키를 사용합니다. 수업용 로컬 환경에서 그 키를 환경 변수 OPENAI_API_KEY에 설정하거나, 노트북과 같은 폴더의 .env에서 읽게 하세요. 키는 노트북에 적지 않습니다. 모델 호출 셀을 다시 실행하면 새 요청이 발생합니다. 새 커널이라면 아래 키 입력 셀을 실행하세요. 입력한 값은 이 커널과 여기서 띄운 채팅 앱에서만 사용합니다.

## 11장 · 메시지 목록으로 대화 이어 가기

한 메시지의 `role`은 말한 주체이고 `content`는 말한 내용입니다. `user`는 사용자의 말, `assistant`는 모델의 답입니다. 다음 질문을 보낼 때는 이전 메시지와 새 메시지를 순서대로 한 목록에 담습니다.

### 첫 답을 받은 뒤 다음 요청의 메시지 조립하기

In [ ]:
previous = [{"role": "assistant", "content": "장소는 201호입니다."}]
message = {"role": "user", "content": "305호로 바꿔 주세요."}
outgoing = previous + [message]
print([item["role"] for item in outgoing])
print(len(previous))

['assistant', 'user']
1


보낼 목록에는 두 역할이 있고, 원래 목록의 길이는 여전히 1입니다. 목록 덧셈만 실행했습니다. 아래의 실제 `send`는 모델 응답이 도착한 뒤에 이력을 갱신합니다.

### 수업 폴더에 호출 함수 준비하기

수업 커널에 필요한 패키지를 설치합니다. 이미 설치되어 있으면 다음 셀로 넘어가세요. 설치 직후 가져오기에 실패하면 커널을 재시작하세요.

In [ ]:
%pip install "openai>=2,<4" "streamlit>=1.40,<2" python-dotenv

In [ ]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")  # 수업 폴더에 키 파일이 있다면 읽습니다.
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("수업용 OpenAI API 키: ")  # 입력한 키는 화면에 표시되지 않습니다.

노트북과 화면 양쪽에서 같은 함수를 쓰도록 다음 셀을 한 번 실행하세요. 첫 줄의 %%writefile은 셀 내용을 공지 도우미 파일에 저장하는 Jupyter 명령입니다. 같은 이름의 파일이 있으면 덮어씁니다. 먼저 instructions를 읽어 보세요. “정정한 항목”과 “그대로 둔 항목”을 구분하면 날짜만 바꾸는 요청의 범위가 명확해집니다.

첫 줄의 `%%writefile notice_chat.py`는 아래 코드를 파일에 쓰는 Jupyter 명령입니다. 이 셀만으로 모델에 질문을 보내지는 않습니다. 다음 셀에서 불러올 `send(history, question)`은 답의 문자열을 반환하면서 전달받은 목록도 바꿉니다. `history + [user]`로 요청용 목록을 따로 만들고, 답이 온 뒤 `extend`로 실제 이력 끝에 두 메시지를 붙이는 순서를 보세요.

In [ ]:
%%writefile notice_chat.py
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()  # 환경에 설정한 키를 읽습니다.
RULE = "소리채집 동아리 공지 도우미입니다. 사용자가 정정한 항목은 최신 내용을 사용하고 그대로 둔 항목은 유지하세요. 모임 정보가 없으면 어떤 정보가 필요한지 질문하세요."

def send(history, question):
    # 이번 입력에 user 역할을 붙이면 앞선 모델 답과 구별할 수 있습니다.
    user = {"role": "user", "content": question}
    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions=RULE,
        input=history + [user],  # 앞선 대화 뒤에 새 질문을 붙여 보냅니다.
        store=False,
    )
    reply = response.output_text
    # 응답을 받은 뒤에만 한 쌍을 기록하므로 실패한 질문은 이력에 남지 않습니다.
    history.extend([user, {"role": "assistant", "content": reply}])
    return reply

### 대화를 한 번에 읽는 연습

날짜를 정정하는 사람은 같은 장소와 시간까지 반복하지 않습니다. 첫 공지에서 ‘5월 21일, 오후 4시, 305호’라고 했고 다음 발화에는 “5월 22일입니다. 나머지는 같습니다”만 있습니다. **가장 늦게 말한 값이 있는 항목만** 바꿔 보세요. 결과는 22일, 오후 4시, 305호입니다. 이제 “아까 말한 곳”과 “지난 채팅에서 말한 곳”이 같은지 생각해 보세요. 이 수업의 프로그램은 이번 대화의 메시지 목록만 보냅니다.

| 시점 | 프로그램이 이번 요청에 넣을 것 | 예상하고 나서 확인할 것 |
|---|---|---|
| 첫 공지 | 사용자 발화 1개 | 원문 날짜와 장소가 맞는가 |
| 날짜 정정 | 첫 발화, 모델의 첫 답, 새 정정 발화 | 바뀔 날짜와 유지할 장소가 구별되는가 |
| 마지막 요약 | 앞서 오간 두 쌍, 마지막 사용자 발화 | 최종 한 문장이 같은 행사에 관한 것인가 |
| 새 대화에서 요약 | 마지막 사용자 발화만 | 구체적인 날짜를 알 근거가 있는가 |

학생 한 명은 사용자, 한 명은 모델 역할을 맡습니다. 앞선 두 종이쪽지가 있는 경우와 마지막 쪽지만 있는 경우로 나눠 답을 써 보세요. 입력을 건네지 않았는데 정답을 예상하는 실수를 이 단계에서 발견할 수 있습니다.

### send(history, question)로 한 턴 보내기

새 목록을 만들고 C4의 첫 발화를 보냅니다. 출력에서 **5월 21일·오후 4시·가온관 305호**를 찾아 보세요. 아래 출력은 2026년 11월 26일 gpt-4.1-mini를 실제 호출한 결과입니다. 답변의 문장과 줄바꿈은 실행할 때마다 달라질 수 있습니다.

In [3]:
from notice_chat import send
history = []  # 이 셀부터 다시 실행하면 빈 대화로 시작합니다.
print(send(history, "모임은 5월 21일 금요일 오후 4시예요. 가온관 305호에서 만나요."))

소리채집 동아리 모임 안내입니다.

일시: 5월 21일 금요일 오후 4시  
장소: 가온관 305호

더 필요한 정보가 있으면 알려주세요!


첫 공지에 대한 답입니다. 이제 장소를 다시 입력하지 않고 날짜만 고칩니다. 첫 셀을 다시 실행하지 말고 바로 아래 셀로 가세요.

In [4]:
print(send(history, "날짜를 정정할게요. 5월 22일 토요일입니다. 시간과 장소는 그대로예요."))

수정된 소리채집 동아리 모임 안내입니다.

일시: 5월 22일 토요일 오후 4시  
장소: 가온관 305호

다른 문의 사항 있으시면 알려주세요!


날짜는 22일로 바뀌고 오후 4시와 305호는 남았습니다. 이제 원문 사실을 다시 쓰지 않고 답변 형식만 지정해 봅니다.

In [5]:
last_question = "최종 모임 일시와 장소를 한 문장으로 정리해 주세요."
print(send(history, last_question))

소리채집 동아리 모임은 5월 22일 토요일 오후 4시에 가온관 305호에서 진행됩니다.


### 목록 슬라이스로 직전 요청에 포함된 메시지 읽기

`send`가 성공하면 기록 끝에 사용자의 질문과 모델의 답을 한 쌍으로 붙입니다. 다음 셀에서 세 번째 요청 직전의 메시지를 확인합니다. 세 번째 질문까지의 목록은 지금의 기록에서 마지막 모델 답 하나를 뺀 값입니다. 전체 글을 출력해 살피는 일이므로 모델 호출은 생기지 않습니다.

In [ ]:
# 방금 받은 마지막 답은 제외하고, 그 요청 때 보낸 메시지만 읽습니다.
for message in history[:-1]:
    print(message["role"], message["content"])

user 모임은 5월 21일 금요일 오후 4시예요. 가온관 305호에서 만나요.
assistant 소리채집 동아리 모임 안내입니다.

일시: 5월 21일 금요일 오후 4시  
장소: 가온관 305호

더 필요한 정보가 있으면 알려주세요!
user 날짜를 정정할게요. 5월 22일 토요일입니다. 시간과 장소는 그대로예요.
assistant 수정된 소리채집 동아리 모임 안내입니다.

일시: 5월 22일 토요일 오후 4시  
장소: 가온관 305호

다른 문의 사항 있으시면 알려주세요!
user 최종 모임 일시와 장소를 한 문장으로 정리해 주세요.


이 출력은 앞 셀에 저장된 실제 공지 대화로 목록 조회만 다시 실행한 결과입니다. 현재 커널에서 받은 모델 답의 표현은 달라질 수 있습니다.

마지막 줄에는 user와 ‘최종 모임 일시와 장소를 한 문장으로 정리해 주세요’가 나와야 합니다. 그 위에서 305호를 누가 처음 알려 주었는지 찾으세요. assistant의 말도 다음 요청에 들어가지만, 모델이 앞서 말한 문장 자체가 새로운 사실을 입증하지는 않습니다. 출력에서 지시가 보이지 않는 이유도 찾아보세요. 공통 지시는 메시지 목록 밖의 `instructions=RULE`로 따로 전송합니다.

### 실습 · 똑같은 마지막 질문을 새 대화에 보내기

기존 history는 그대로 보관하세요. 다른 빈 목록을 만들고 방금 쓴 last_question만 보내세요. 답에 305호가 있다면 그 방 번호를 알려 준 메시지가 실제 요청에 들어갔는지 확인하세요. 출력 문장을 바탕으로 동료에게 “초기화하면 어떤 정보가 사라지는가?”를 설명하세요.

In [ ]:
fresh = []
# send(fresh, last_question)의 답을 출력하세요. 직전 대화의 장소와 시각을 되물을 수 있습니다.
# 호출 뒤 fresh 안에 이번 질문과 답만 들어 있는지 role을 순서대로 출력하세요.

<details><summary>모범 예시와 확인 기준</summary><pre><code>fresh = []
print(send(fresh, last_question))
print([item[&quot;role&quot;] for item in fresh])

# 수업에서 실제로 받은 응답
# 최종 모임 일시와 장소 정보를 알려주시면 한 문장으로 정리해 드리겠습니다.</code></pre><p>두 번째 출력은 [&#39;user&#39;, &#39;assistant&#39;]입니다. 새 목록에는 지난 공지가 한 건도 없습니다. 모임 정보를 되묻는 답이 근거에 맞습니다. 기존 history를 또 넘겼다면 새 대화가 시작되지 않습니다.</p></details>

### 정정이 겹치면 어느 안내를 남겨야 할까요?

[운영진 문서](../documents/sources/editing-desk.md) M1, M0, M2를 읽습니다. M1에는 집결지가 지난번과 같다고 되어 있고, M0에서 지난 장소가 솔빛관임을 알 수 있습니다. 뒤이어 M2가 ‘이번 집결지를 다온관 서문으로 변경’합니다. 이번에는 단지 마지막 지명을 고르는 문제가 아닙니다. **같은 산책을 정정한 문서인지**, **바뀐 항목이 집결지인지 행사 날짜인지**부터 확인해야 합니다.

1. 기존 C4와 구분되는 빈 목록을 하나 만들어 M0와 M1을 함께 보냅니다. 집결지와 시작 시간을 묻고 실제 답을 복사합니다.
2. M2 본문만 다음 발화로 보냅니다. “참가자와 진행자가 새로 알아야 할 점을 나눠 주세요”라고 물으세요.
3. 새 목록에 M2와 같은 질문만 넣어 봅니다. 이번 집결지는 M2에서 답할 수 있습니다. 그러나 시작 시각까지 같은 근거로 확정할 수 있는지 살핍니다.

수정 전과 후의 모든 사실을 표로 만드는 코드는 필요 없습니다. 화면에서 **새로 정해진 것**, **앞선 문서에서 유지한 것**, **아직 모르는 것** 한 가지씩 짚어 보세요.

<details><summary>정정 결과의 예시 펼치기</summary><p>이전 문서가 함께 있으면 참가자는 6월 14일 오전 10시에 다온관 서문으로 갑니다. 진행자는 오전 9시 30분에 이름표와 우산 보관함을 준비합니다. 장소와 추가 준비물은 M2, 시작 시각과 30분 전 도착 지시는 M1이 근거입니다.</p><p>M2만 받은 새 대화라면 집결지와 추가 준비물은 답할 수 있어도 시작 시각은 알 수 없습니다. “나머지는 앞선 공지와 같습니다”는 앞선 공지의 값이 무엇인지 알려 주지 않습니다. 방 이름이나 시각을 맞혔는지만으로 대화가 제대로 이어졌다고 판정하지 마세요.</p></details>

In [ ]:
# C4와 별도의 실험입니다. M0·M1을 보낸 뒤 M2를 같은 목록에 이어 보냅니다.
walk_history = []
# print(send(walk_history, "M0와 M1의 본문을 각각 제목과 함께 넣으세요."))
# print(send(walk_history, "M2 본문과 확인하고 싶은 질문을 넣으세요."))

<details><summary>두 대화의 입력을 나누는 방법</summary><pre><code># 첫 두 발화를 같은 목록에 넣고, 똑같은 질문을 별도의 빈 목록에도 보냅니다.
old_information = &quot;지난번 집결지는 솔빛관 1층 안내대 앞입니다. 이번 산책은 6월 14일 오전 10시, 집결지는 지난번과 같습니다.&quot;
correction = &quot;6월 14일 집결지를 다온관 서문으로 변경합니다. 다른 항목은 같습니다. 집결지와 시작 시각을 알려 주세요.&quot;
walk_history = []
print(send(walk_history, old_information))
print(send(walk_history, correction))
print(send([], correction))</code></pre><p>마지막 호출은 집결지 변경 자체를 알고 있지만 오전 10시라는 값은 받은 적이 없습니다. 여기서 비교할 것은 답의 길이가 아니라 각 값에 대한 근거입니다.</p></details>

## 12장 · 질문이 들어올 때만 응답하는 채팅 화면

노트북에서는 원하는 셀만 골라 실행했습니다. Streamlit은 사용자가 입력을 보내거나 체크박스를 바꾸면 앱 파일을 위에서부터 다시 실행합니다. 파일 맨 위에 history = []를 쓰면 이전 대화가 사라집니다. st.session_state는 같은 브라우저 탭에서 다시 실행할 때에도 값을 보관하므로 대화 목록을 여기에 둡니다.

화면에서 할 일은 세 가지입니다. 먼저 기존 메시지를 그립니다. 새 입력을 확인합니다. 입력이 들어온 실행에서만 send를 부릅니다. <mark>지난 답을 다시 화면에 그리는 동안에는 모델에 다시 묻지 않습니다.</mark>

### st.chat_input과 session_state로 입력한 문장 다시 그리기

첫 셀은 관찰용 앱을 `echo_app.py`로 저장합니다. 다음 셀은 노트북과 같은 Python으로 그 앱을 8502번 포트에서 띄웁니다. 모델 키 없이 입력과 화면 재실행만 확인할 수 있습니다.

앱을 열면 목록은 비어 있습니다. ‘사과’를 한 번 보낸 뒤 체크박스를 두 번 눌러 보세요. 체크박스를 바꿀 때마다 화면은 다시 그려지지만 같은 말풍선은 한 개여야 합니다.

In [ ]:
%%writefile echo_app.py
import streamlit as st

# 처음 연 탭에서만 빈 목록을 만듭니다.
if "notes" not in st.session_state:
    st.session_state.notes = []

text = st.chat_input("공지를 한 줄 입력하세요")
if text:
    st.session_state.notes.append(text)  # 새 입력이 있을 때만 보관합니다.

st.checkbox("화면 다시 실행해 보기")
for note in st.session_state.notes:
    st.chat_message("user").write(note)  # 예전에 보낸 문장도 화면에 다시 그립니다.

In [ ]:
import subprocess
import sys
# 완성 앱이 쓸 8501을 비워 두고 관찰용 화면은 8502에서 실행합니다.
echo_server = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "echo_app.py",
                                "--server.port", "8502", "--server.headless", "true"])

브라우저에서 http://localhost:8502 를 열고 “오전 10시”, “장소 변경”을 차례로 보내세요. 체크박스를 누른 뒤에도 메시지는 두 개입니다. `if text:` 안에서만 목록에 새 말을 추가하기 때문입니다.

이제 파일을 쓰는 셀에서 목록을 비우는 대입문을 조건 밖으로 옮겨 보세요. 파일을 다시 저장한 뒤 같은 입력으로 차이를 확인합니다. 다음 셀은 실험을 마친 뒤 실행해 관찰용 서버를 종료합니다.

In [ ]:
echo_server.terminate()
echo_server.wait()

### st.chat_input의 반환값으로 호출 시점 정하기

학생이 말을 보내면 `st.chat_input(...)`이 그 문자열을 반환합니다. 새 입력이 없는 재실행에서는 `None`을 받습니다. 채팅창에 “안녕”을 보낸 실행에서는 `if question:` 안에서 한 번 요청하고, 체크박스만 누른 실행에서는 저장된 말만 다시 그립니다.

### 화면 파일 만들기

다음 셀에서 위쪽은 기존 메시지 표시, 아래쪽 if question은 새 입력 처리입니다. 채팅 입력을 보내지 않은 실행에서는 question에 질문이 없으므로 그 안을 건너뜁니다. 체크박스는 모델 호출 횟수와 대화 쌍 수를 확인하기 위한 것입니다.

화면의 모델 요청 수는 앱이 send를 시작한 횟수입니다. 호출에 실패하면 실패 안내가 뜨고, 질문과 답을 저장하는 줄에는 도달하지 않습니다. 이때 요청 수와 성공한 대화 쌍 수가 달라도 정상입니다.

`st.chat_input`은 전송한 문장이 있으면 그 문자열을 돌려줍니다. 새 입력이 없는 재실행에서는 아래의 `if question:` 안으로 들어가지 않습니다. `st.chat_message(...).write(...)`는 이미 있는 글을 말풍선에 표시합니다. 체크박스만 눌렀을 때 화면은 다시 그려져도 `send`를 한 번 더 부르지 않아야 합니다.

In [ ]:
%%writefile notice_app.py
import streamlit as st
from openai import APIError
from notice_chat import send

st.title("소리채집 동아리 공지 도우미")
if "messages" not in st.session_state:
    st.session_state.messages = []
    st.session_state.calls = 0

# 초기화 과제: 이곳에 새 대화 버튼과 초기화 동작을 추가하세요.
show_counts = st.checkbox("실행 정보 보기")
history = st.session_state.messages
for message in history:
    st.chat_message(message["role"]).write(message["content"])

question = st.chat_input("공지나 후속 질문을 입력하세요")
if question:
    st.session_state.calls += 1  # 보내기를 누른 입력만 집계합니다.
    st.chat_message("user").write(question)
    try:
        reply = send(history, question)
    except APIError:
        # 응답이 없어 저장하지 않았으므로 같은 질문을 다시 보낼 수 있습니다.
        st.error("응답을 받지 못했습니다. 연결과 키를 확인한 뒤 질문을 다시 보내 주세요.")
    else:
        st.chat_message("assistant").write(reply)

if show_counts:
    st.caption(f"모델 요청 {st.session_state.calls}건 · 저장된 대화 {len(history) // 2}쌍")

### 내 컴퓨터에서 채팅창 열기

아래 셀을 한 번 실행하고 브라우저에서 http://localhost:8501 을 여세요. 노트북의 Python과 같은 환경으로 서버를 시작합니다. 주소가 열리지 않으면 셀 아래에 나온 서버 오류를 확인하세요. 8501을 다른 수업 앱이 쓰고 있다면 여기의 포트 번호와 브라우저 주소를 함께 바꾸세요.

In [ ]:
import subprocess
import sys

server = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "notice_app.py",
                           "--server.port", "8501", "--server.headless", "true"])

### 실습 · 두 번 보내고 체크박스만 바꾸기

‘실행 정보 보기’를 켜세요. 첫 입력으로 아래 C1 공지를 보내고, 두 번째 입력으로 “진행자는 몇 시까지 가야 하나요?”라고 물으세요. 그다음 체크박스를 껐다가 켜세요. 활동지에는 첫 입력, 두 번째 입력, 체크박스 조작 뒤의 요청 건수와 대화 쌍 수를 적습니다.

5월 21일 금요일 오후 4시에 가온관 201호에서 소리채집 동아리 첫 모임을 엽니다. 신입 회원은 이어폰을 챙겨 오세요. 진행자는 모임 20분 전에 도착해 녹음기를 책상마다 한 대씩 놓아 주세요.

<details><summary>모범 예시와 확인 기준</summary><pre><code>첫 입력  → 요청 1건 · 저장된 대화 1쌍
후속 질문 → 요청 2건 · 저장된 대화 2쌍
체크박스 다시 켜기 → 요청 2건 · 저장된 대화 2쌍</code></pre><p>공지의 오후 4시에서 20분을 빼면 오후 3시 40분입니다. 체크박스만 만졌는데 건수가 늘었다면 send가 if question 안에 있는지 확인하세요. 답은 한 번인데 메시지가 두 번 보인다면 기존 기록을 그리는 위치와 새 답을 그리는 위치를 살펴보세요.</p></details>

### 증상으로 고칠 곳 좁히기

| 실험에서 본 현상 | 먼저 볼 값 | 고쳐서 다시 시험할 부분 |
|---|---|---|
| 체크박스를 켜자 기존 메시지가 사라짐 | 빈 목록을 만드는 줄이 매번 실행되는가 | 세션에 목록이 없을 때만 빈 목록을 만듦 |
| 두 번째 질문으로 정정한 날짜를 묻자 처음 듣는다고 답함 | 화면에 그린 과거 발화가 실제 호출의 input에도 들어가는가 | 목록을 읽는 코드와 `send(history, question)`에 넘긴 객체를 대조 |
| 한 번 보냈는데 같은 질문·답이 두 쌍 남음 | `send` 안에서도 기록하고 앱에서도 기록하는가 | 저장을 수행하는 위치를 하나로 모은 뒤 새 대화에서 확인 |
| 요청 수만 늘고 답이 남지 않음 | 호출 시도와 성공한 응답을 어떻게 세는가 | 오류 메시지와 마지막 성공한 대화를 보고 같은 질문을 한 번 재전송 |

브라우저에서 새 탭에 앱 주소를 직접 입력해 별도의 세션을 열어 보세요. 이전 탭은 열어 둡니다. 새 탭에 공지가 없는 상태에서 구체적 날짜를 말했는지 보고, 원래 탭의 메시지와 요청 수는 바뀌었는지도 보세요. 시크릿 창을 사용해도 됩니다. 세션 상태는 실행 중 대화를 담는 공간이므로 장기 보존할 자료는 활동지에 직접 기록합니다.

### 개인 실습 · 두 사람이 같은 앱을 쓰면

A탭의 사용자는 “A동, 오전 9시”를, B탭의 사용자는 “B동, 오후 2시”를 첫 공지로 보냅니다. 두 번째 입력은 양쪽 모두 “어느 동 몇 시인가요?”입니다. 두 탭이 같은 URL을 열어도 각 사용자가 받은 답은 달라야 합니다. 어느 시점에 무엇이 모델에 들어갔는지 예상한 다음 실행하세요.

각 탭에서 사용자 질문 두 건과 답 두 건을 확인하고, A탭에서만 체크박스를 바꾼 뒤 B탭 호출 수를 읽어 보세요. 만약 A탭 정답에 B동이 섞였다면 입력 원문을 잘못 붙였는지, 이력이 세션 바깥의 공용 변수로 옮겨졌는지 순서대로 살핍니다. 실험이 끝나면 다음 장으로 가져갈 “실패 증상 → 먼저 확인할 값” 한 쌍을 활동지에 작성합니다.

<details><summary>두 탭의 모범 관찰 결과</summary><p>성공한 호출에서 A탭은 A동 오전 9시, B탭은 B동 오후 2시의 근거가 있습니다. 각 탭은 요청 2건·대화 2쌍을 가집니다. A의 체크박스 조작은 어느 쪽에도 새 질문을 추가하지 않습니다. 문장에 부연 설명이 있더라도 원문에 없는 건물을 지어냈는지는 따로 판정하세요.</p></details>

## 13장 · 대화 이력과 요청 수 초기화하기

`st.session_state`에 저장한 대화 목록을 빈 리스트로 바꾸면 다음 요청은 새 대화에서 시작합니다. 같은 후속 질문을 초기화 전과 후에 보내고, 그 질문에 필요한 앞 메시지가 어느 쪽에 들어 있는지 확인하세요.

대화를 초기화할 때는 다음 요청의 `input`으로 보낼 목록을 비워야 합니다. `st.session_state.messages`에 지난 대화가 남아 있다면 말풍선을 잠시 감춰도 모델에는 그 내용이 전달됩니다. 초기화 뒤 장소 없이 “어디서 만나죠?”라고 물어 원래 장소가 요청에 포함될 수 있는지 검사하세요.

### 실습 · 새 대화 버튼 완성하기

`st.button("새 대화")`는 버튼을 누른 실행에서 참입니다. 버튼 조건 안에서만 목록과 횟수를 초기값으로 되돌리세요. 조건 밖에서 초기화하면 다음 메시지를 보낼 때도 이력이 사라집니다.

화면 파일을 만드는 셀에서 ‘초기화 과제’ 주석 자리를 고치세요. 버튼을 눌렀을 때 메시지 목록과 요청 수를 둘 다 초기화하세요. 기존 기록을 그리기 전에 처리해야 그 실행부터 빈 화면이 보입니다. 화면 파일 셀을 다시 실행해 저장하고 브라우저에서 변경된 앱을 다시 실행하세요. Streamlit이 변경을 감지해 Rerun을 물으면 그 버튼을 누르세요.

검사는 짧은 질문 한 건으로 시작하세요. 답을 받은 뒤 ‘새 대화’를 누르면 요청 수가 0이고 말풍선도 사라져야 합니다. 그다음 질문을 한 번 보내면 요청 수는 1입니다.

<details><summary>모범 예시와 확인 기준</summary><pre><code>if st.button(&quot;새 대화&quot;):
    st.session_state.messages = []  # 다음 질문에 보낼 이력도 비웁니다.
    st.session_state.calls = 0</code></pre><p>체크박스를 켰을 때 요청 0건·대화 0쌍이 보여야 합니다. 화면에서 글만 지우고 messages를 비우지 않으면 다음 호출에 지난 내용이 들어갑니다.</p></details>

### 동료가 처음 보는 C5로 시연하기

1. 빈 대화에서 C5 원문과 “신입 회원용으로 두 문장으로 정리해 줘”를 한 메시지로 보냅니다.
2. “해야 할 일과 마감을 남겨 한 문장으로 줄여 줘”만 보냅니다. 원문과 나란히 두고 5월 28일 금요일 오전 11시, 동아리 게시판 제출, 파일 제목에 녹음 장소 표시가 남았는지 확인하세요.
3. 새 대화를 누른 뒤 2번 문장만 보내세요. 원문을 다시 달라는 답을 받았는지 기록합니다. 모델이 구체적인 마감을 제시했다면 실제로 보낸 정보와 대조하세요.

C5 원문: 5월 28일 금요일 오전 11시까지 지난 모임에서 녹음한 소리를 동아리 게시판에 올려 주세요. 파일 제목에 녹음 장소를 넣어 주세요. 다음 주 모임에서 함께 들을 파일은 진행자가 두 개 고릅니다.

<details><summary>모범 예시와 확인 기준</summary><pre><code>학생이 원문을 읽고 만든 한 문장 예시
5월 28일 금요일 오전 11시까지 파일 제목에 녹음 장소를 적어 지난 모임의 녹음 파일을 동아리 게시판에 올려 주세요.</code></pre><p>첫 두 요청은 2건·2쌍입니다. 버튼 직후는 0건·0쌍이고, 마지막 질문 후에는 1건·1쌍입니다. 날짜를 모르는 새 대화에서 원문을 요청하는 응답도 이 실습의 성공 사례입니다.</p></details>

### 인수 실습 · 시설 담당자가 혼자 실행할 수 있는가

이제 동아리 팀에서 도서관 시설 담당자에게 프로그램을 넘깁니다. [시설 원문](../documents/sources/facility-notices.md)을 열고 S1을 사용하세요. 호출 함수 파일을 만드는 셀에서 공통 지시를 편집합니다. 예를 들어 “시설 안내를 읽고 이용자 행동을 안내합니다. 바뀐 항목만 최신 정정으로 교체하고, 다른 정보는 유지하세요. 중단한 서비스의 범위를 원문으로 확인하세요.”처럼 업무에 맞는 부분만 고칩니다. 수정한 파일 셀을 다시 실행해 저장하세요. 서버를 시작한 노트북의 새 셀에서 `server.terminate()`와 `server.wait()`를 차례로 실행한 뒤, 12장의 서버 시작 셀을 한 번 더 실행합니다. 새 서버에서 수정된 지시를 읽습니다.

사용자는 도서관 전체가 닫히는지, 반납함만 쓸 수 없는지 알고 싶습니다. 독자와 상황을 정한 첫 요청을 S1 원문과 함께 보냅니다. 다음 발화에는 S2만 보내고 “오전 11시 반에 반납할 곳과 영향을 받는 서비스를 한 문장으로 다시 써 줘”라고 물으세요. 대화가 길어져도 시작 시각, 종료 시각, 대체 반납 장소, 정상 운영 범위를 혼동하지 않는지 확인합니다.

**새 자료 인수 시험**. 다른 팀은 H1·H2의 본문과 자신이 쓴 질문으로 새 대화를 시작합니다. “원래 예약일에도 장소를 옮겨야 하나요?”라는 질문에 답한 뒤, 버튼으로 초기화해서 그 질문만 다시 보냅니다. 첫 실행부터 질문을 다시 입력하는 과정을 기록합니다. 수정이 필요하면 무엇을 바꿀지 결정하세요. 잘못된 시간을 대답했다면 근거와 공통 지시를 검토하고, 정상적인 답이 두 번 저장되었다면 화면 코드를 검토해야 합니다.

<details><summary>시설·방송실 인수 예시 열기</summary><p>S1과 S2를 받은 대화에서는 7월 8일 오전 9시부터 오후 1시까지 동문 무인 반납함을 점검합니다. 오전 11시 반에는 1층 안내 데스크에서 반납할 수 있으며 열람실과 예약 도서 수령은 정상입니다. ‘도서관 전체 운영 중단’은 범위를 넓힌 오류입니다.</p><p>H1과 H2의 경우 8월 2일 방송실은 정상 이용할 수 있습니다. 수리 시간은 8월 3일 오후 2~4시로 옮겼고 해당 시간의 대체 장소는 본관 1층 소리실입니다. 초기화한 뒤 ‘원래 예약일’만 묻는다면 시설이나 날짜를 확인하는 질문이 가능합니다. 실제 답을 옮긴 뒤 앞선 발화가 들어갔는지, 원문 구절로 답할 수 있는지를 따로 표시하세요.</p></details>

### 다음 수업으로 넘기는 질문

지금 챗봇은 사용자가 붙인 글을 읽습니다. “지난 산책 안내에서 원래 집결지는 어디였죠?”라고 새 대화에 물었을 때 M0의 내용을 답할 자료는 어디에 있을까요? 사용자가 찾아서 붙여 주거나, 프로그램이 문서를 찾아 질문과 함께 보내 주어야 합니다. 먼저 손으로 관련 문단 한 개를 골라 표시하고, 그 문단과 질문을 함께 보내 답할 수 있는지 확인해 보세요. 14장에서는 단어 하나로 프롬프트·모델·파서를 연결합니다. 15장부터 PDF를 읽고, 19장에서 검색한 문단을 질문과 함께 그 체인에 보냅니다.

[Streamlit의 세션 상태 설명](https://docs.streamlit.io/develop/concepts/architecture/session-state) · [채팅 화면 만들기](https://docs.streamlit.io/develop/tutorials/chat-and-llm-apps/build-conversational-apps)

수업을 마친 뒤 다음 셀을 실행하세요. `terminate()`는 노트북에서 띄운 8501번 서버에 종료를 요청하고, `wait()`는 그 프로세스가 끝날 때까지 기다립니다. 이번 노트북과 마지막으로 확인한 입력을 함께 보관하세요.

In [ ]:
server.terminate()
server.wait()